# SGLang Training Lab: Performance Optimization, Benchmarking & Profiling

This training lab walks you through practical performance tuning with SGLang. You will learn:

1. **Launch & Configure** — How to start an SGLang server with performance-oriented flags (tensor parallelism, KV cache dtype, etc.)
2. **Benchmarking** — Measure throughput and latency under different batch sizes and concurrency levels
3. **Profiling** — Use PyTorch Profiler via SGLang's HTTP API to capture GPU/CPU traces and identify bottlenecks
4. **Performance Tuning** — Key knobs you can turn to squeeze more performance out of your deployment

We will take `Qwen/Qwen3-30B-A3B-Instruct-2507` model as example. The cookbook for this model can be found [here](https://cookbook.sglang.io/autoregressive/Qwen/Qwen3).

---


## 0. Prerequisites

Before running this notebook, make sure:
- This notebook is running inside [SGLang v0.5.9 docker container](https://hub.docker.com/layers/lmsysorg/sglang/v0.5.9/images/sha256-f337dfb36971becc98d12768f356af3c6c12ba57c9aebede0e6948da5ad37da7). The Python kernel is set to `/usr/bin/python`
- SGLang with version 0.5.9 has been installed in the docker image, which can be checked with `pip list | grep sglang`.
- You have GPU(s) available (this lab assumes multi-GPU for TP=2). It can be checked by running `nvidia-smi`.

In [ ]:
!which python
!pip list | grep sglang
!nvidia-smi

## 1. Configuration

In [ ]:
import os
import json
import time
import subprocess
import requests

# Download the model from Hugging Face
MODEL_PATH = "Qwen/Qwen3-30B-A3B-Instruct-2507"

# Server settings
HOST = "0.0.0.0"
PORT = 30000
BASE_URL = f"http://127.0.0.1:{PORT}"

# Profiling output directory
PROFILE_DIR = "/dli/task"
os.environ["SGLANG_TORCH_PROFILER_DIR"] = PROFILE_DIR
os.makedirs(PROFILE_DIR, exist_ok=True)

print(f"Model path:  {MODEL_PATH}")
print(f"Server URL:  {BASE_URL}")
print(f"Profile dir: {PROFILE_DIR}")

## 2. Launch the SGLang Server

We launch the server with several performance-oriented flags:

| Flag | Purpose |
|---|---|
| `--tp 2` | Tensor parallelism across 2 GPUs — splits model weights to fit large models |
| `--trust-remote-code` | Allow loading custom model code from the checkpoint |

> **Note**: Quantization and KV cache dtype depend on the model. See Section 5 for additional tuning knobs.

In [ ]:

# Build the launch command
# Adjust flags based on your model — this is a general-purpose template.

LAUNCH_CMD = f"""
SGLANG_TORCH_PROFILER_DIR={PROFILE_DIR} \
python3 -m sglang.launch_server \
    --model-path {MODEL_PATH} \
    --trust-remote-code \
    --host {HOST} \
    --port {PORT} \
    --tp 2
""".strip()

print("Launch command:")
print(LAUNCH_CMD)
print()

When launching `Qwen3-30B-A3B-Instruct-2507` on H100, the following optimization will be applied by default:
- Continuous Batching
- Overlap Scheduler (See [v0.4](https://lmsys.org/blog/2024-12-04-sglang-v0-4/) blog for details)
- Flash Attention 3 kernel for attention computation
- Triton fused MoE kernel
- Cuda graph on decode batches, with maximum captured batch size being 256 by default
- Flashinfer Allreduce + RMSNorm fused kernel: [Doc](https://docs.flashinfer.ai/generated/flashinfer.comm.AllReduceFusionOp.html)

In [ ]:
# Launch the server as a background process.
# The server takes a few minutes to load model weights and warm up.
# Here we redirect the output to a log file for tracking.
log_file = open("server_output.log", "w")
server_process = subprocess.Popen(
    LAUNCH_CMD,
    shell=True,
    stdout=log_file,
    stderr=log_file,
    text=True,
)

In [ ]:
# Wait for the server to be ready (polls the /health endpoint)
from sglang.utils import wait_for_server

print("Waiting for server to start (this may take several minutes for large models)...")
wait_for_server(BASE_URL, process=server_process, timeout=600)
print("Server is ready!")

### Quick Sanity Check

Verify the server is responding correctly.

In [ ]:
import openai

client = openai.Client(base_url=f"{BASE_URL}/v1", api_key="EMPTY")

response = client.chat.completions.create(
    model=MODEL_PATH,
    messages=[{"role": "user", "content": "Please introduce yourself"}],
    temperature=0,
    max_tokens=128,
)

print("Model response:", response.choices[0].message.content)
print(f"Usage: {response.usage.prompt_tokens} prompt tokens, {response.usage.completion_tokens} completion tokens")

In [ ]:
# Check server info — shows the active configuration
server_info = requests.get(f"{BASE_URL}/get_server_info").json()
print(json.dumps(server_info, indent=2))

---

## 3. Benchmarking

SGLang ships with `sglang.bench_one_batch_server`, a latency benchmarking tool that sends a single static batch to the server. This gives you precise, repeatable measurements of prefill and decode latency at a given batch size.

We will measure:

- **Prefill latency** — time to process the input prompt
- **Decode latency** — time per output token during generation
- **End-to-End latency** — total time for the full request
- **Throughput** — tokens per second at a fixed batch size

### Key `bench_one_batch_server` parameters

| Parameter | Description |
|---|---|
| `--model None` | Use `None` when connecting to an already-running server |
| `--base-url` | URL of the running SGLang server |
| `--dataset-name random` | Generate random prompts (no external dataset needed) |
| `--batch-size` | Number of requests in the batch (controls parallelism) |
| `--input-len` | Input sequence length |
| `--output-len` | Output sequence length |
| `--result-filename` | Save raw results as JSONL |
| `--profile` | Enable PyTorch profiler during the benchmark |

### 3.1 Warmup

The first run after server start may trigger JIT kernel compilation. Always warm up and flush the cache before your actual measurements.

In [ ]:
def flush_cache():
    """Flush the KV cache on the server to ensure a clean state."""
    resp = requests.get(f"{BASE_URL}/flush_cache")
    print(f"Cache flushed: {resp.text.strip()}")

def run_benchmark(batch_size, input_len=1024, output_len=1024,
                  result_filename="result.jsonl", extra_args=""):
    """Run sglang.bench_one_batch_server and return the output."""
    cmd = (
        f"python3 -m sglang.bench_one_batch_server "
        f"--model None "
        f"--base-url {BASE_URL} "
        f"--dataset-name random "
        f"--batch-size {batch_size} "
        f"--input-len {input_len} "
        f"--output-len {output_len} "
        f"--result-filename {result_filename} "
        f"--skip-warmup "
        f"{extra_args}"
    )
    print(f"Running: {cmd}\n")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

In [ ]:
# Warmup run — triggers JIT kernel compilation on first invocation.
# We run a small batch directly (without --skip-warmup) so the internal warmup runs too.
flush_cache()
print("Running warmup (this also triggers JIT kernel compilation)...")
warmup_cmd = (
    f"python3 -m sglang.bench_one_batch_server "
    f"--model None --base-url {BASE_URL} "
    f"--dataset-name random --batch-size 1 "
    f"--input-len 1024 --output-len 16 "
    f"--result-filename warmup.jsonl"
)
warmup_result = subprocess.run(warmup_cmd, shell=True, capture_output=True, text=True)
print(warmup_result.stdout)
if warmup_result.returncode != 0:
    print("Warmup had non-zero exit (flush_cache race is harmless, check stderr):")
    print(warmup_result.stderr[-500:] if len(warmup_result.stderr) > 500 else warmup_result.stderr)
print("Warmup complete.")

### 3.2 Benchmark: Batch Size 1 (Latency-Oriented)

With `--batch-size 1`, a single request is in-flight. This measures single-request latency — useful for interactive / chat use cases.

In [ ]:
flush_cache()
run_benchmark(batch_size=1, result_filename="bench_bs1.jsonl")

### 3.3 Benchmark: Batch Size 64 (Throughput-Oriented)

With `--batch-size 64`, 64 requests run concurrently in a single batch. This saturates the GPU and measures throughput — useful for batch processing and serving under load.

In [ ]:
flush_cache()
run_benchmark(batch_size=64, result_filename="bench_bs64.jsonl")

### 3.4 Comparing Results

Load and compare the benchmark outputs side-by-side.

In [ ]:
def load_bench_results(filepath):
    """Load all results from a bench_one_batch_server JSONL output file."""
    results = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                results.append(json.loads(line))
    return results

def print_bench_summary(result, label=""):
    print(f"{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Batch size:              {result.get('batch_size', 'N/A')}")
    print(f"  Input len:               {result.get('input_len', 'N/A')}")
    print(f"  Output len:              {result.get('output_len', 'N/A')}")
    print(f"  E2E Latency:             {result.get('latency', 0):.2f} s")
    print(f"  TTFT (last):             {result.get('last_ttft', 0):.2f} s")
    print(f"  Input throughput:        {result.get('input_throughput', 0):.2f} tok/s")
    print(f"  Output throughput:       {result.get('output_throughput', 0):.2f} tok/s")
    print(f"  Overall throughput:      {result.get('overall_throughput', 0):.2f} tok/s")
    print(f"  Overall throughput per request: {(result.get('overall_throughput', 0) / result.get('batch_size')):.2f} tok/s")
    print()

try:
    bs1_results = load_bench_results("bench_bs1.jsonl")
    bs64_results = load_bench_results("bench_bs64.jsonl")
    if bs1_results:
        print_bench_summary(bs1_results[-1], "Batch Size 1 (Latency-Oriented)")
    if bs64_results:
        print_bench_summary(bs64_results[-1], "Batch Size 64 (Throughput-Oriented)")
except FileNotFoundError as e:
    print(f"Benchmark file not found: {e}")
    print("Run the benchmark cells above first.")

### 3.5 Sweep Across Batch Sizes

To build an intuition for how throughput scales with batch size.

In [ ]:
def sweep_batch_sizes(prefix=""):
    batch_sizes = [1, 4, 16, 64, 256]
    sweep_results = []

    for bs in batch_sizes:
        flush_cache()
        run_benchmark(
            batch_size=bs,
            result_filename=f"{prefix}_bench_sweep_bs{bs}.jsonl",
        )
        try:
            results = load_bench_results(f"{prefix}_bench_sweep_bs{bs}.jsonl")
            if results:
                sweep_results.append((bs, results[-1]))
        except Exception:
            pass

    print(f"\n{'Batch Size':>12} {'Throughput tok/s':>15} {'Throughput per request tok/s':>25} {'TTFT (s)':>10} {'Latency (s)':>13}")
    print("-" * 68)
    for bs, r in sweep_results:
        print(f"{bs:>12} {r.get('overall_throughput', 0):>15.2f} {(r.get('overall_throughput', 0) / r.get('batch_size')):>25.2f} {r.get('last_ttft', 0):>10.2f} {r.get('latency', 0):>13.2f}")
    return sweep_results

In [ ]:
def draw_curve(sweep_results):
    import matplotlib.pyplot as plt

    overall_tp = [r.get('overall_throughput', 0) for _, r in sweep_results]
    per_req_tp = [r.get('overall_throughput', 0) / r.get('batch_size') for _, r in sweep_results]
    bs_labels  = [str(bs) for bs, _ in sweep_results]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(overall_tp, per_req_tp, marker='o', linewidth=2, markersize=8)

    for x, y, label in zip(overall_tp, per_req_tp, bs_labels):
        ax.annotate(f"bs={label}", (x, y), textcoords="offset points",
                    xytext=(8, 8), fontsize=9)

    ax.set_xlabel("Overall Throughput (tok/s)", fontsize=12)
    ax.set_ylabel("Throughput per Request (tok/s)", fontsize=12)
    ax.set_title("Overall Throughput vs Per-Request Throughput", fontsize=14)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
# We can draw a "Pareto Curve" for the throughput and interactivity (throughput per request).
# Usually when the batch size increases, the throughput increases, but the throughput per request decreases.
%matplotlib inline
sweep_results = sweep_batch_sizes()
draw_curve(sweep_results)

---

## 4. Profiling with PyTorch Profiler

SGLang integrates PyTorch Profiler and exposes HTTP endpoints to start/stop profiling on a running server. This lets you capture detailed GPU/CPU traces while the server is under load.

### How it works

1. **`POST /start_profile`** — Starts capturing a trace. You can specify how many forward steps to capture.
2. The server runs workload while profiling is active.
3. **`POST /stop_profile`** — Stops profiling (or it auto-stops after `num_steps`).
4. Trace files are written to `SGLANG_TORCH_PROFILER_DIR` in Chrome Trace format.
5. Open traces in [Perfetto UI](https://ui.perfetto.dev/) or `chrome://tracing`.

### Key parameters for `/start_profile`

| Parameter | Description |
|---|---|
| `output_dir` | Where to save traces (defaults to `SGLANG_TORCH_PROFILER_DIR`) |
| `num_steps` | Number of forward steps to profile (auto-stops) |
| `start_step` | Skip N warmup steps before recording |
| `activities` | `["CPU", "GPU"]` by default. Can add `"MEM"` for memory profiling |
| `merge_profiles` | Merge traces from multiple TP/DP ranks into one file |
| `profile_by_stage` | Profile prefill and decode separately |

### 4.1 Profile While Running a Benchmark

The simplest approach: use `bench_one_batch_server --profile` which automatically starts/stops the profiler around the benchmark run.

In [ ]:
flush_cache()

profile_cmd = (
    f"python3 -m sglang.bench_one_batch_server "
    f"--model None "
    f"--base-url {BASE_URL} "
    f"--dataset-name random "
    f"--batch-size 1 "
    f"--input-len 1024 "
    f"--output-len 1024 "
    f"--profile"
)

print(f"Running profiled benchmark...")
print(f"Command: {profile_cmd}\n")
result = subprocess.run(profile_cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# List generated trace files
import glob

trace_files = sorted(glob.glob(os.path.join(PROFILE_DIR, "**/*.trace.json.gz"), recursive=True))
if trace_files:
    print(f"Found {len(trace_files)} trace file(s):")
    for f in trace_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {f}  ({size_mb:.1f} MB)")
else:
    print(f"No trace files found in {PROFILE_DIR}")
    print("Make sure SGLANG_TORCH_PROFILER_DIR is set on the server side as well.")

### 4.2 Profile Using the HTTP API (Programmatic Control)

For more fine-grained control, you can start/stop profiling via HTTP while sending your own workload.

In [ ]:
flush_cache()

# Start profiling: skip 2 warmup steps, then capture 10 steps
profile_request = {
    "output_dir": PROFILE_DIR,
    "start_step": 2,
    "num_steps": 10,
    "activities": ["CPU", "GPU"],
    "merge_profiles": True,
}

print("Starting profiler...")
print(f"Config: {json.dumps(profile_request, indent=2)}")

In [ ]:
import concurrent.futures

def send_chat_requests(n=10):
    """Send concurrent chat requests to generate load."""
    client = openai.Client(base_url=f"{BASE_URL}/v1", api_key="EMPTY")
    results = []
    for i in range(n):
        resp = client.chat.completions.create(
            model=MODEL_PATH,
            messages=[{"role": "user", "content": f"Write a short paragraph about topic {i}."}],
            max_tokens=256,
            temperature=0.7,
        )
        results.append(resp)
    return results

# Start profiling and send load concurrently
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    profile_future = executor.submit(
        requests.post,
        f"{BASE_URL}/start_profile",
        json=profile_request,
    )
    # Give the profiler a moment to initialize
    time.sleep(1)
    load_future = executor.submit(send_chat_requests, 10)

    profile_resp = profile_future.result()
    load_results = load_future.result()

print(f"Profile response: {profile_resp.status_code} - {profile_resp.text}")
print(f"Completed {len(load_results)} chat requests during profiling.")

In [ ]:
# List updated trace files
trace_files = sorted(glob.glob(os.path.join(PROFILE_DIR, "**/*.trace.json.gz"), recursive=True))
print(f"\nTrace files in {PROFILE_DIR}:")
for f in trace_files[-5:]:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {f}  ({size_mb:.1f} MB)")

print("\n>> Open these files in https://ui.perfetto.dev/ to visualize the trace.")

### 4.3 Profile Using `sglang.profiler` (Convenience Module)

SGLang provides a standalone profiler module that wraps the HTTP API.

In [ ]:
from sglang.profiler import run_profile

# Make sure there is active workload when profiling!
# In practice, run this while bench_serving or your application is sending requests.

# Start profiler for 5 forward steps
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    profiler_future = executor.submit(
        run_profile,
        url=BASE_URL,
        num_steps=5,
        activities=["CPU", "GPU"],
        output_dir=PROFILE_DIR,
        merge_profiles=True,
    )
    time.sleep(0.5)
    load_future = executor.submit(send_chat_requests, 5)

    trace_dir = profiler_future.result()
    _ = load_future.result()

print(f"\nTrace saved to: {trace_dir}")

### 4.4 Reading & Interpreting Traces

After profiling, open the `.trace.json.gz` file at [https://ui.perfetto.dev/](https://ui.perfetto.dev/).

**What to look for:**

- **GPU kernel timeline** — Are there gaps between kernels? Gaps = CPU overhead or synchronization stalls.
- **Long-running kernels** — Which CUDA kernels dominate decode time? (e.g., attention, MoE, all-reduce)
- **CPU overhead** — Scheduling, tokenization, and Python overhead between GPU kernel launches. Kernel fusion can help with CPU overhead.
- **NCCL communication** — In multi-GPU (TP) setups, look for communication (all reduce, all gather...) time between layers. Notes that on some ranks the communication time might be extremely long, but it is actually caused by waiting rather than slow kernel time.

**Tips:**
- Use `--disable-cuda-graph` when launching the server if you want to see the full Python call stack in the trace (CUDA graph captures hide individual kernel launches).
- Keep traces small by limiting `--num-prompts` and output length.

---

## 5. Performance Tuning Guide

### 5.1 Key Server Configuration Knobs

| Knob | Flag | When to Use |
|---|---|---|
| **Tensor Parallelism** | `--tp N` | Model too large for one GPU, or to increase decode throughput |
| **Data Parallelism Attention** | `--enable-dp-attention --dp N` | Prefer for throughput when memory allows (better throughput and worse latency vs TP, works better for GQA models) |
| **Quantization** | `--quantization fp8` (online quantization) | Reducing model weight memory, enabling faster kernels and larger batch sizes. For offline quantization please use fp8 version of model (`Qwen/Qwen3-30B-A3B-Thinking-2507-FP8`) |
| **Memory Fraction** | `--mem-fraction-static 0.88` | Tune to maximize KV cache pool; leave ~5-8 GB for activations |
| **Schedule Conservativeness** | `--schedule-conservativeness 0.3` | Lower = more aggressive batching (use when `token usage < 0.9` but queue is not empty). Use a larger value if you see requests being retracted frequently. |
| **CUDA Graph Max BS** | `--cuda-graph-max-bs 512` | When memory is enough, increasing CUDA graph capture range helps for higher decode batch sizes |
| **Chunked Prefill Size** | `--chunked-prefill-size 4096` | Lower to reduce OOM during prefill; Higher for better hardware utilization and speed. Trades prefill speed for stability |
| **Max Concurrency** | `--max-running-requests` | Controls the number of running requests. Higher concurrency can reach better throughput, but at risk of retraction or OOM.  |

### 5.2 Monitoring Server Health

Look for these log patterns when the server is under load:

```
Decode batch. #running-req: 233, #token: 370959, token usage: 0.82, cuda graph: True, gen throughput (token/s): 4594.01, #queue-req: 317
```

| Metric | Target | Action |
|---|---|---|
| `token usage` | > 0.9 | If low + queue > 0: decrease `--schedule-conservativeness` |
| `#queue-req` | 100 - 2000 | If 0: server fast enough to handle all the requests. If very large: server cannot satisfy the request from clients, and the server should tune all the knobs above |
| `gen throughput` | Maximize | Tune all the knobs above |

In [ ]:
# Check current server metrics
try:
    metrics = requests.get(f"{BASE_URL}/get_server_info").json()
    print("Current server configuration:")
    for key in ["tp_size", "dp_size", "mem_fraction_static", "schedule_conservativeness",
                "chunked_prefill_size", "kv_cache_dtype", "context_len"]:
        if key in metrics:
            print(f"  {key}: {metrics[key]}")
except Exception as e:
    print(f"Could not fetch server info: {e}")

In [ ]:
def terminate_server(server_process):
    # Terminate the server when done
    from sglang.utils import terminate_process

    terminate_process(server_process)

    print("Server terminated.")

In [ ]:
terminate_server(server_process)

## 5.3 Examples

### FP8 Quantization
Qwen-3 officially provides `Qwen/Qwen3-30B-A3B-Thinking-2507-FP8`, where some of the attention weights and all the MoE weights are quantized to FP8_E4M3 format. With quantized weights, the attention/MoE kernels can run at a higher speed with negligible output quality.

In [ ]:
# Try launching with offline quantized model
QUANTIZED_MODEL_PATH = "Qwen/Qwen3-30B-A3B-Thinking-2507-FP8"
LAUNCH_CMD_QUANTIZED = f"""
python3 -m sglang.launch_server \
    --model-path {QUANTIZED_MODEL_PATH} \
    --trust-remote-code \
    --host {HOST} \
    --port {PORT} \
    --tp 2
""".strip()

print("Launch command:")
print(LAUNCH_CMD_QUANTIZED)
print()

In [ ]:
log_file = open("server_output.log", "w")
os.environ["SGLANG_JIT_DEEPGEMM_FAST_WARMUP"] = "1"
server_process = subprocess.Popen(
    LAUNCH_CMD_QUANTIZED,
    shell=True,
    stdout=log_file,
    stderr=log_file,
    text=True,
)

In [ ]:
# Wait for the server to be ready (polls the /health endpoint)
from sglang.utils import wait_for_server

print("Waiting for server to start (this may take minutes for DeepGEMM warmup)...")
wait_for_server(BASE_URL, process=server_process, timeout=600)
print("Server is ready!")

In [ ]:
# First sweep one time for warmup
warmup_sweep_results = sweep_batch_sizes()

In [ ]:
# Draw Pareto Curve for quantized model
%matplotlib inline
sweep_results = sweep_batch_sizes(prefix="quantized")
draw_curve(sweep_results)

In [ ]:
terminate_server(server_process)

### Spectulative Decoding with EAGLE-3

Speculative decoding is a strong weapon for optimization performance on small batches. Here we applies the [EAGLE-3](https://arxiv.org/abs/2503.01840) algorithm, with the draft models weight pretrained by [SpecForge](https://github.com/sgl-project/SpecForge) framework.

In [ ]:
# Try launching with EAGLE-3 speculative decoding method
EAGLE_3_PATH = "lmsys/SGLang-EAGLE3-Qwen3-30B-A3B-Instruct-2507-SpecForge-Nex"
LAUNCH_CMD_SPEC = f"""
python3 -m sglang.launch_server \
    --model-path {MODEL_PATH} \
    --speculative-algorithm EAGLE3 \
    --speculative-draft-model-path {EAGLE_3_PATH} \
    --speculative-num-steps 3 \
    --speculative-eagle-topk 1 \
    --speculative-num-draft-tokens 4 \
    --trust-remote-code \
    --host {HOST} \
    --port {PORT} \
    --tp 2
""".strip()

print("Launch command:")
print(LAUNCH_CMD_SPEC)
print()

In [ ]:
log_file = open("server_output.log", "w")
server_process = subprocess.Popen(
    LAUNCH_CMD_SPEC,
    shell=True,
    stdout=log_file,
    stderr=log_file,
    text=True,
)

In [ ]:
# Wait for the server to be ready (polls the /health endpoint)
from sglang.utils import wait_for_server

print("Waiting for server to start (this may take several minutes for large models)...")
wait_for_server(BASE_URL, process=server_process, timeout=600)
print("Server is ready!")

In [ ]:
# Draw Pareto Curve for Speculative Decoding
%matplotlib inline
sweep_results = sweep_batch_sizes(prefix="Spec")
draw_curve(sweep_results)

In [ ]:
terminate_server(server_process)

---

## Quick Reference: CLI Commands

For running outside this notebook.

### Launch Server
```bash
export SGLANG_TORCH_PROFILER_DIR=/root/sglang/profile_log

python3 -m sglang.launch_server \
    --model-path YOUR_MODEL_PATH \
    --trust-remote-code --tp 4
```

### Benchmarking
```bash
# Warmup
curl http://127.0.0.1:30000/flush_cache
python3 -m sglang.bench_one_batch_server \
    --model None --base-url http://localhost:30000 \
    --dataset-name random --batch-size 1 \
    --input-len 1024 --output-len 1024

# Batch size 1
curl http://127.0.0.1:30000/flush_cache
python3 -m sglang.bench_one_batch_server \
    --model None --base-url http://localhost:30000 \
    --dataset-name random --batch-size 1 \
    --input-len 1024 --output-len 1024 \
    --result-filename bench_bs1.jsonl

# Batch size 64
curl http://127.0.0.1:30000/flush_cache
python3 -m sglang.bench_one_batch_server \
    --model None --base-url http://localhost:30000 \
    --dataset-name random --batch-size 64 \
    --input-len 1024 --output-len 1024 \
    --result-filename bench_bs64.jsonl
```

### Profiling
```bash
# Profile with bench_one_batch_server
python3 -m sglang.bench_one_batch_server \
    --model None --base-url http://localhost:30000 \
    --dataset-name random --batch-size 1 \
    --input-len 1024 --output-len 1024 --profile

# Profile with HTTP API
curl -X POST http://127.0.0.1:30000/start_profile \
    -H "Content-Type: application/json" \
    -d '{"num_steps": 10, "start_step": 3, "merge_profiles": true}'

# Profile with sglang.profiler
python3 -m sglang.profiler --num-steps 10 --cpu --gpu --merge-profiles
```

### View Traces
Open `.trace.json.gz` files at [https://ui.perfetto.dev/](https://ui.perfetto.dev/)